# Caso de estudio: Rotación de empleados (RR. HH.)
**Introducción a Machine Learning – Universidad de Cundinamarca**
**Autor:** Harold Steven Rueda Lara

**Objetivo:** predecir si un empleado renunciará (1) o permanecerá (0) en la empresa, aplicando Regresión Logística y modelos de clasificación (Árbol de Decisión, Random Forest, KNN, SVM y Naive Bayes), y analizar los resultados con accuracy, precisión, recall, F1-score y matriz de confusión.

**Datos y modelos:** se usan el dataset y la configuración de modelos del script del profesor (`C2_CasoRRHH.py`). Además se incluyen los ejercicios del módulo 4 de la guía práctica (separación de datos, `StandardScaler`, Regresión Logística, matriz de confusión y reporte de clasificación) y los gráficos del módulo 3 (Matplotlib).

## 1. Librerías

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import (train_test_split, cross_val_score,
                                     RepeatedStratifiedKFold, cross_validate)
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report,
                             ConfusionMatrixDisplay)

## 2. Dataset
Datos del script del profesor: 20 empleados con edad, salario (en miles), años en la empresa, nivel de satisfacción, horas extra y la variable objetivo `renuncia` (0 = permanece, 1 = renuncia).

In [ ]:
data = {
    "edad": [25, 28, 35, 40, 30, 45, 32, 29, 50, 38,
             27, 33, 41, 36, 31, 48, 26, 39, 42, 34],
    "salario": [1800, 2000, 3500, 4500, 2200, 5000, 2800, 2100, 6000, 4000,
                1900, 3000, 4700, 3600, 2500, 5500, 1700, 4200, 4800, 3200],
    "años_empresa": [1, 2, 5, 8, 2, 10, 4, 2, 15, 7,
                     1, 4, 9, 6, 3, 12, 1, 8, 10, 5],
    "satisfaccion": [2, 3, 7, 8, 3, 9, 5, 2, 9, 7,
                     2, 6, 8, 6, 4, 9, 1, 7, 8, 5],
    "horas_extra": [15, 12, 5, 4, 14, 3, 8, 13, 2, 6,
                    16, 7, 4, 6, 10, 3, 18, 5, 4, 8],
    "renuncia": [1, 1, 0, 0, 1, 0, 0, 1, 0, 0,
                 1, 0, 0, 0, 1, 0, 1, 0, 0, 0],
}
df = pd.DataFrame(data)
df.head()

### 2.1 Exploración

In [ ]:
print("Filas y columnas:", df.shape)
print("\nValores nulos por columna:")
print(df.isnull().sum())
print("\nResumen estadístico:")
print(df.describe().round(2))
print("\nDistribución de la variable objetivo:")
print(df["renuncia"].value_counts().rename({0: "permanece", 1: "renuncia"}))

In [ ]:
print("Promedio de cada variable según si el empleado renuncia o no:")
df.groupby("renuncia").mean().round(2).rename(index={0: "permanece", 1: "renuncia"})

### 2.2 Gráficos (módulo 3 de la guía)
Figura con `plt.subplots(1, 2)`: histograma de la satisfacción y diagrama de dispersión entre satisfacción y horas extra, con títulos, etiquetas, leyendas y rejilla.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].hist(df.loc[df.renuncia == 0, "satisfaccion"], bins=range(1, 11), alpha=0.7,
             label="Permanece", color="gray", edgecolor="black")
axes[0].hist(df.loc[df.renuncia == 1, "satisfaccion"], bins=range(1, 11), alpha=0.7,
             label="Renuncia", color="black", edgecolor="white")
axes[0].set_title("Distribución del nivel de satisfacción")
axes[0].set_xlabel("Satisfacción (1 a 10)"); axes[0].set_ylabel("Número de empleados")
axes[0].legend(); axes[0].grid(True, linestyle="--", alpha=0.5)

for valor, etiqueta, color, marca in [(0, "Permanece", "gray", "o"), (1, "Renuncia", "black", "x")]:
    sub = df[df.renuncia == valor]
    axes[1].scatter(sub["satisfaccion"], sub["horas_extra"], label=etiqueta, color=color, marker=marca, s=60)
axes[1].set_title("Satisfacción vs. horas extra")
axes[1].set_xlabel("Satisfacción (1 a 10)"); axes[1].set_ylabel("Horas extra")
axes[1].legend(); axes[1].grid(True, linestyle="--", alpha=0.5)

plt.tight_layout(); plt.show()

## 3. Variables X e y y división de datos
70 % para entrenamiento y 30 % para prueba, con `stratify` para mantener la proporción de renuncias. Con solo 20 empleados, el conjunto de prueba tiene 6.

In [ ]:
X = df[["edad", "salario", "años_empresa", "satisfaccion", "horas_extra"]]
y = df["renuncia"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y)

print("Entrenamiento:", X_train.shape[0], "empleados | Prueba:", X_test.shape[0], "empleados")
print("Renuncias en prueba:", int(y_test.sum()), "de", len(y_test))

## 4. Ejercicio de la guía (módulo 4): estandarización y Regresión Logística
Se estandarizan las variables con `StandardScaler` (media 0 y desviación 1), se ajusta la Regresión Logística y se obtienen la matriz de confusión y el reporte de clasificación.

In [ ]:
scaler = StandardScaler()
X_train_esc = scaler.fit_transform(X_train)   # el escalador aprende solo de entrenamiento
X_test_esc = scaler.transform(X_test)

reg_log = LogisticRegression(max_iter=1000)
reg_log.fit(X_train_esc, y_train)
y_pred_rl = reg_log.predict(X_test_esc)

print("Matriz de confusión:")
print(confusion_matrix(y_test, y_pred_rl))
print("\nReporte de clasificación:")
print(classification_report(y_test, y_pred_rl, zero_division=0))

print("Coeficientes (variables estandarizadas):")
print(pd.Series(reg_log.coef_[0], index=X.columns).round(3).sort_values())

## 5. Comparación de los seis modelos
Misma configuración del script del profesor (sin estandarizar, `random_state=42`). Se calculan accuracy, precisión, recall, F1-score y validación cruzada de 5 particiones.

In [ ]:
modelos = {
    "Regresión Logística": LogisticRegression(max_iter=1000),
    "Árbol de Decisión": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=3),
    "SVM": SVC(kernel="linear"),
    "Naive Bayes": GaussianNB(),
}

resultados, matrices = [], {}
for nombre, modelo in modelos.items():
    modelo.fit(X_train, y_train)
    pred = modelo.predict(X_test)
    matrices[nombre] = confusion_matrix(y_test, pred)
    cv = cross_val_score(modelo, X, y, cv=5, scoring="accuracy")
    resultados.append({
        "Modelo": nombre,
        "Accuracy": accuracy_score(y_test, pred),
        "Precisión": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "F1-score": f1_score(y_test, pred, zero_division=0),
        "Validación cruzada": cv.mean(),
        "Accuracy entrenamiento": accuracy_score(y_train, modelo.predict(X_train)),
    })

tabla = pd.DataFrame(resultados).set_index("Modelo").round(3)
tabla

### 5.1 Matrices de confusión

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(13, 8))
for ax, (nombre, cm) in zip(axes.ravel(), matrices.items()):
    ConfusionMatrixDisplay(cm, display_labels=["Permanece", "Renuncia"]).plot(ax=ax, cmap="Greys", colorbar=False)
    ax.set_title(nombre)
plt.tight_layout(); plt.show()

print("TN = permanece y se predijo permanece | FP = permanece pero se predijo renuncia")
print("FN = renuncia pero se predijo permanece | TP = renuncia y se detectó\n")
for nombre, cm in matrices.items():
    tn, fp, fn, tp = cm.ravel()
    print(f"{nombre:22s} TN={tn}  FP={fp}  FN={fn}  TP={tp}")

### 5.2 Gráfico comparativo y mejor modelo

In [ ]:
tabla[["Accuracy", "Precisión", "Recall", "F1-score"]].plot(
    kind="bar", figsize=(11, 5), colormap="Greys", edgecolor="black")
plt.title("Comparación de modelos en el conjunto de prueba")
plt.ylabel("Valor"); plt.ylim(0, 1.1); plt.grid(True, axis="y", linestyle="--", alpha=0.5)
plt.xticks(rotation=25, ha="right"); plt.legend(loc="lower right"); plt.tight_layout(); plt.show()

max_f1 = tabla["F1-score"].max()
empatados = tabla.index[tabla["F1-score"] == max_f1].tolist()
print(f"F1-score máximo: {max_f1}")
print(f"Modelos con ese F1-score ({len(empatados)}): {', '.join(empatados)}")

## 6. Robustez: validación cruzada repetida
El conjunto de prueba tiene solo 6 empleados, así que un error de un solo empleado cambia mucho las métricas. Para comparar de forma más estable se repite 20 veces una validación cruzada de 5 particiones estratificadas (100 evaluaciones por modelo) y se reporta el promedio y la desviación estándar.

In [ ]:
cv_rep = RepeatedStratifiedKFold(n_splits=5, n_repeats=20, random_state=42)
filas = []
for nombre, modelo in modelos.items():
    r = cross_validate(modelo, X, y, cv=cv_rep,
                       scoring={"accuracy": "accuracy", "precision": "precision",
                                "recall": "recall", "f1": "f1"})
    filas.append({
        "Modelo": nombre,
        "Accuracy": r["test_accuracy"].mean(),
        "Precisión": r["test_precision"].mean(),
        "Recall": r["test_recall"].mean(),
        "F1 promedio": r["test_f1"].mean(),
        "F1 desv. est.": r["test_f1"].std(),
    })
tabla_cv = pd.DataFrame(filas).set_index("Modelo").round(3)
tabla_cv.sort_values("F1 promedio", ascending=False)

## 7. ¿Qué variables influyen más en la renuncia?
Importancia de variables del Random Forest y coeficientes de la Regresión Logística con datos estandarizados (un coeficiente positivo aumenta la probabilidad de renuncia; uno negativo la reduce).

In [ ]:
rf = modelos["Random Forest"]
imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values()
coef = pd.Series(reg_log.coef_[0], index=X.columns).sort_values()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
imp.plot(kind="barh", ax=axes[0], color="gray", edgecolor="black")
axes[0].set_title("Importancia de variables (Random Forest)"); axes[0].set_xlabel("Importancia")
coef.plot(kind="barh", ax=axes[1], color="gray", edgecolor="black")
axes[1].axvline(0, color="black", linewidth=0.8)
axes[1].set_title("Coeficientes (Regresión Logística estandarizada)"); axes[1].set_xlabel("Coeficiente")
for ax in axes: ax.grid(True, axis="x", linestyle="--", alpha=0.5)
plt.tight_layout(); plt.show()

## 8. Sobreajuste (overfitting)
Comparación entre el accuracy en entrenamiento y en prueba. Una diferencia grande indica que el modelo se aprendió los datos de memoria.

In [ ]:
comp = tabla[["Accuracy entrenamiento", "Accuracy"]].rename(columns={"Accuracy": "Accuracy prueba"})
comp["Diferencia"] = (comp["Accuracy entrenamiento"] - comp["Accuracy prueba"]).round(3)
comp

## 9. Pregunta 8 de la guía: ¿es bueno este modelo?
Un modelo con Accuracy = 0,90, Precisión = 0,70, Recall = 0,95 y F1-score = 0,80. Se verifica que el F1 sea coherente con la precisión y el recall.

In [ ]:
p, r = 0.70, 0.95
print(f"F1 = 2*P*R/(P+R) = {2*p*r/(p+r):.3f}")

## 10. Cómo se interpretan las métricas en este caso
- **Accuracy:** porcentaje de empleados bien clasificados (los que permanecen y los que renuncian).
- **Precisión:** de los empleados que el modelo dice que renunciarán, cuántos realmente renuncian.
- **Recall:** de los empleados que realmente renuncian, cuántos detecta el modelo. Es la métrica que más interesa a RR. HH. si se quiere anticipar las renuncias.
- **F1-score:** balance entre precisión y recall.
- **Matriz de confusión:** muestra TN, FP, FN y TP, es decir, qué tipo de errores comete cada modelo.